# How to generate a combined utility/privacy report?

### Create a combined report of the metrics, whether they are utility or privacy metrics. /!\ Only for the summary.

Assume that the synthetic data is already generated \
Based on the Wisconsin Breast Cancer Dataset

In [1]:
# Standard library
import sys
import tempfile

sys.path.append("..")

# 3rd party packages
import pandas as pd

# Local packages
import config
import utils.draw
from metrics.report import Report

## Load the real and synthetic WBCD datasets

In [2]:
df_real = {}
df_real["train"] = pd.read_csv(
    "../data/" + config.WBCD_DATASET_TRAIN_FILEPATH.stem + ".csv"
)
df_real["test"] = pd.read_csv(
    "../data/" + config.WBCD_DATASET_TEST_FILEPATH.stem + ".csv"
)
df_real["train"].shape

(359, 10)

### Choose the synthetic dataset

In [3]:
df_synth = {}  # generated by Synthpop ordered here
df_synth["train"] = pd.read_csv("../results/data/2023-07-31_Synthpop_359samples.csv")
df_synth["test"] = pd.read_csv("../results/data/2023-07-31_Synthpop_90samples.csv")
df_synth["test"].shape

(90, 10)

## Configure the metadata dictionary

### The continuous and categorical variables need to be specified, as well as the variable to predict for the future learning task

In [4]:
metadata = {
    "continuous": [
        "Clump_Thickness",
        "Uniformity_of_Cell_Size",
        "Uniformity_of_Cell_Shape",
        "Marginal_Adhesion",
        "Single_Epithelial_Cell_Size",
        "Bland_Chromatin",
        "Normal_Nucleoli",
        "Mitoses",
        "Bare_Nuclei",
    ],
    "categorical": ["Class"],
    "variable_to_predict": "Class",
}

## Generate the report

Some metrics will not be computed since the only categorical variable is the variable to predict 

In [5]:
parameters = {  # see the notebooks utility_report and privacy_report for more details
    "cross_learning": True,
    "num_repeat": 3,
    "use_gpu": True,
    "sampling_frac": 0.2,
}

In [6]:
report = Report(
    dataset_name="Wisconsin Breast Cancer Dataset",
    df_real=df_real,
    df_synthetic=df_synth,
    metadata=metadata,
    figsize=(8, 6),  # will be automatically adjusted for larger or longer figures
    random_state=0,  # for reproducibility purposes
    report_folderpath=None,  # load computed utility and/or privacy reports if available
    report_filename=None,  # the name of the computed report (without extension nor utility/privacy) if available
    metrics=None,  # list of the metrics to compute. Can be utility or privacy metrics. If not specified, all the metrics are computed.
    params=parameters,  # the dictionary containing the parameters for both utility and privacy reports
)

In [7]:
report.compute()

## Get the summary report as a pandas dataframe

In [8]:
report.specification()

----- Wisconsin Breast Cancer Dataset -----
Contains:
    - 359 instances in the train set,
    - 90 instances in the test set,
    - 10 variables, 9 continuous and 1 categorical.


In [9]:
df_summary = report.summary()

In [10]:
by = ["name", "objective", "min", "max"]
df_summary.groupby(by).apply(lambda x: x.drop(by, axis=1).reset_index(drop=True))

alias  \
name                                          objective min max                            
Categorical Consistency                       max       0   1.0 0             cat_consis   
Categorical Statistics                        max       0   1.0 0              cat_stats   
                                                                1              cat_stats   
Classification                                min       0   1.0 0                classif   
Continuous Consistency                        max       0   1.0 0            cont_consis   
Continuous Statistics                         min       0   inf 0             cont_stats   
                                                                1             cont_stats   
Cross Classification                          min       0   1.0 0          cross_classif   
Cross Regression                              min       0   inf 0              cross_reg   
DCR                                           max       0   1.0 0                    dcr   
                                                            inf 0                    dcr   
Distinguishability                            min       0   1.0 0                   dist   
                                                                1                   dist   
                                                                2                   dist   
                                                                3                   dist   
FScore                                        min       0   inf 0                 fscore   
Feature Importance                            min       0   inf 0            feature_imp   
Hellinger Categorical Univariate Distance     min       0   1.0 0     hell_cat_univ_dist   
Hellinger Continuous Univariate Distance      min       0   1.0 0    hell_cont_univ_dist   
KL Divergence Categorical Univariate Distance min       0   inf 0   kl_div_cat_univ_dist   
KL Divergence Continuous Univariate Distance  min       0   inf 0  kl_div_cont_univ_dist   
Pairwise Correlation Difference               min       0   inf 0                    pcd   

                                                                                     submetric  \
name                                          objective min max                                  
Categorical Consistency                       max       0   1.0 0                 within_ratio   
Categorical Statistics                        max       0   1.0 0             support_coverage   
                                                                1           frequency_coverage   
Classification                                min       0   1.0 0              diff_real_synth   
Continuous Consistency                        max       0   1.0 0                 within_ratio   
Continuous Statistics                         min       0   inf 0           median_l1_distance   
                                                                1              iqr_l1_distance   
Cross Classification                          min       0   1.0 0              diff_real_synth   
Cross Regression                              min       0   inf 0              diff_real_synth   
DCR                                           max       0   1.0 0   nndr_5th_percent_synthreal   
                                                            inf 0    dcr_5th_percent_synthreal   
Distinguishability                            min       0   1.0 0               prediction_mse   
                                                                1          prediction_mse_real   
                                                                2         prediction_mse_synth   
                                                                3               prediction_auc   
FScore                                        min       0   inf 0                 diff_f_score   
Feature Importance                            min       0   inf 0  diff_permutation_importance   
Hellinger Ca

## Save and load the report

In [11]:
with tempfile.TemporaryDirectory() as temp_dir:
    report.save(savepath=temp_dir, filename="report")  # save
    new_report = Report(report_folderpath=temp_dir, report_filename="report")  # load